# LumiLearn 模型部署全流程

## 为什么需要了解部署流程？

LumiLearn 是一个自训练的 GPT-2 架构中文教育模型。从 Python 训练脚本到浏览器终端，中间经历了：
- PyTorch checkpoint → GGUF 格式转换
- GGUF 量化（FP32 → Q4_K_M）
- Ollama 模型注册
- Flask 终端服务搭建

本 notebook 带你走通每一步，理解背后的原理。

## 环境准备
- Python 3.10+
- PyTorch
- `gguf` 库 (`pip install gguf`)
- Ollama (https://ollama.com)
- Flask + requests

## 部署全流程分几步？
- 理解 LumiLearn 模型架构（GPT-2 + BPE）
- 检查 PyTorch checkpoint 结构
- 导出 GGUF：权重 + tokenizer 写入二进制格式
- 量化：从 FP32 压缩到 Q4_K_M
- Ollama 注册：Modelfile 配置
- 终端服务：Flask 代理 + 浏览器 UI

## 认识 LumiLearn 模型架构

LumiLearn 使用 GPT-2 风格 Transformer：
- Pre-LN（LayerNorm 在 Attention/FFN 之前）
- GELU 激活函数
- 绝对位置编码（Learned Positional Embedding）
- 权重绑定（lm_head 和 token_embedding 共享权重）

关键参数：

In [ ]:
# LumiLearn V5 配置
V5_CONFIG = {
    "vocab_size": 8000,       # BPE subword tokenizer
    "hidden_size": 384,       # 隐藏维度
    "num_layers": 8,          # Transformer 层数
    "num_heads": 8,           # 注意力头数
    "ff_dim": 1024,           # Feed-Forward 维度
    "max_seq_len": 384,       # 最大序列长度
}

# 参数量估算
def estimate_params(config):
    d = config["hidden_size"]
    V = config["vocab_size"]
    L = config["num_layers"]
    ff = config["ff_dim"]

    # Embedding (共享权重)
    emb = V * d
    pos = config["max_seq_len"] * d

    # 每层 Transformer: QKV投影 + Output投影 + FFN
    attn = 4 * d * d          # Q, K, V, O
    ffn = 2 * d * ff          # 两个线性层

    # LayerNorm
    ln = 2 * (2 * d)          # pre-LN + post-LN, each has gamma + beta

    total = emb + pos + L * (attn + ffn + ln)
    return total

params = estimate_params(V5_CONFIG)
print(f"总参数量: {params:,}  (~{params/1e6:.1f}M)")
print(f"FP32 占用: {params * 4 / 1024 / 1024:.1f} MB")
print(f"Q4_K_M 占用: {params * 0.5 / 1024 / 1024:.1f} MB")

## BPE Tokenizer：从文字到数字

LumiLearn 使用 BPE (Byte-Pair Encoding) tokenizer，词表大小 8000。
相比字符级 tokenizer，BPE 可以将 "三角形面积公式" 从 6 个 token 压缩到 2-3 个。

GGUF 格式中 tokenizer 存储为 `tokenizer.ggml.tokens` 字段，
这是 Ollama 加载模型时第一个读取的元数据。

In [ ]:
# 演示 BPE tokenizer 的压缩效果
def simulate_bpe_vs_char(text):
    """模拟 BPE vs 字符级 tokenizer 的 token 数量对比"""
    char_tokens = len(text)
    # BPE 约能压缩 50%（实际取决于词表覆盖）
    bpe_ratio = 0.5
    bpe_tokens = max(1, int(char_tokens * bpe_ratio))
    return char_tokens, bpe_tokens

examples = ["三角形面积公式", "勾股定理", "一元二次方程求根公式"]
for text in examples:
    char_t, bpe_t = simulate_bpe_vs_char(text)
    print(f"{text:20s}  char={char_t:2d}  BPE~{bpe_t:2d}  压缩率={bpe_t/char_t:.0%}")

## GGUF 格式：模型的"集装箱"

GGUF (GPT-Generated Unified Format) 是 llama.cpp 生态的模型文件格式。
一个 GGUF 文件包含：

1. **Header**: 魔数 `GGUF` + 版本号
2. **Metadata**: 键值对（架构类型、参数、tokenizer 词表等）
3. **Tensor data**: 所有权重张量

我们使用 Python 的 `gguf` 库来读写 GGUF 文件，避免手动二进制操作的 Bug。

In [ ]:
# 检查 gguf 库是否可用
try:
    import gguf
    print(f"gguf 库版本: {gguf.__version__ if hasattr(gguf, '__version__') else 'installed'}")
except ImportError:
    print("需要安装: pip install gguf")

# GGUF 文件结构示意
print("""
GGUF 文件结构:
┌──────────────────────────┐
│  Magic: 0x46554747 (GGUF) │
│  Version: 3              │
│  Tensor count: N         │
│  Metadata count: M       │
├──────────────────────────┤
│  Metadata:               │
│    general.architecture  │
│    gpt2.context_length   │
│    gpt2.embedding_length │
│    tokenizer.ggml.tokens │ ← 最关键的字段！
│    tokenizer.ggml.eos_id │
│    ...                   │
├──────────────────────────┤
│  Tensor data:            │
│    token_embd.weight     │
│    blk.0.attn_q.weight   │
│    blk.0.attn_k.weight   │
│    ...                   │
└──────────────────────────┘
""")

## 检查 PyTorch Checkpoint

训练完成后，checkpoint 保存在 `outputs/LumiLearn-v5*/checkpoints/`。
我们来看看 checkpoint 里有什么。

In [ ]:
import os
import torch

# 查找 checkpoint
def find_checkpoint(base_dir="../outputs"):
    for d in sorted(os.listdir(base_dir)):
        if d.startswith("LumiLearn-v5") and os.path.exists(f"{base_dir}/{d}/checkpoints"):
            ckpt_dir = f"{base_dir}/{d}/checkpoints"
            for f in sorted(os.listdir(ckpt_dir)):
                if f.endswith(".pt"):
                    return f"{ckpt_dir}/{f}"
    return None

ckpt_path = find_checkpoint()
if ckpt_path:
    print(f"找到 checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cpu")
    print(f"Keys: {list(ckpt.keys())[:10]}...")
    if "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
        print(f"权重数量: {len(sd)}")
        # 显示前几个权重的形状
        for i, (k, v) in enumerate(sd.items()):
            if i >= 5:
                break
            print(f"  {k:40s} shape={list(v.shape)}")
else:
    print("未找到 checkpoint，需要先训练模型")
    print("运行: python train.py")

## 量化：FP32 → Q4_K_M

FP32 每个权重 4 字节。Q4_K_M 量化后每个权重约 0.5 字节。
对 ~21M 参数的模型：FP32 ~80MB，Q4_K_M ~10MB。

量化本质是把连续的浮点数映射到离散的整数区间：

$$w_q = \text{round}\left(\frac{w - \text{min}}{\text{scale}}\right)$$

Q4_K_M 使用 4-bit 量化 + 每个 block 有独立的 scale/min。

In [ ]:
import numpy as np

# 模拟量化过程
def simulate_quantization(weights, bits=4):
    """模拟 K-quant 量化"""
    w_min, w_max = weights.min(), weights.max()
    scale = (w_max - w_min) / (2**bits - 1)

    # 量化
    w_q = np.round((weights - w_min) / scale).astype(np.int8)

    # 反量化
    w_deq = w_q.astype(np.float32) * scale + w_min

    # 误差
    mse = np.mean((weights - w_deq) ** 2)

    # 压缩率
    original_bits = weights.nbytes * 8
    quantized_bits = w_q.nbytes * 8

    return w_deq, mse, quantized_bits / original_bits

# 生成模拟权重
np.random.seed(42)
original = np.random.randn(1000).astype(np.float32) * 0.1

dequantized, mse, ratio = simulate_quantization(original, bits=4)
print(f"原始大小: {original.nbytes} bytes")
print(f"量化比例: {ratio:.1%}")
print(f"MSE: {mse:.6f}")
print(f"前5个权重对比:")
for i in range(5):
    print(f"  FP32={original[i]:+.4f}  →  Q4={dequantized[i]:+.4f}")

## Ollama Modelfile 配置

Modelfile 是 Ollama 的模型描述文件，类似 Dockerfile：
- `FROM`: 指向 GGUF 文件
- `PARAMETER`: 推理参数（temperature, top_p, num_ctx 等）
- `TEMPLATE`: 对话模板（ChatML 格式）
- `SYSTEM`: 系统提示词

In [ ]:
# 生成 Modelfile 内容
MODELFILE_TEMPLATE = '''FROM ./lumilearn-v5-q4km.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER num_ctx 384
PARAMETER num_predict 256
PARAMETER repeat_penalty 1.1
PARAMETER stop "<eos>"
PARAMETER stop "###"

SYSTEM """你是 LumiLearn (灵学) AI教育助手，专注于中国K-12教育领域的知识讲解与答疑。
你的能力涵盖数学、物理、化学、语文、英语等学科。
"""

TEMPLATE """{{ .System }}

### 问题
{{ .Prompt }}

### 回答
"""
'''

print("生成的 Modelfile:")
print(MODELFILE_TEMPLATE)

# 关键参数说明
print("\n关键参数:")
print("  num_ctx=384:    因为 LumiLearn max_seq_len=384")
print("  num_predict=256: 每次最多生成 256 tokens")
print("  temperature=0.7: 教育场景,保持准确性")

## 终端服务架构

最终的部署架构如下：

```
浏览器                           远程服务器 (192.168.2.xx)
┌──────────┐                    ┌─────────────────────┐
│          │  HTTP :18080       │  Flask :18080        │
│ 浏览器   │ ────────────────→  │  (lumiterm_server)  │
│          │                    │         │            │
│          │                    │         │ localhost  │
│          │                    │         ▼ :11434     │
│          │                    │  Ollama              │
│          │                    │  lumilearn-v5:real   │
└──────────┘                    └─────────────────────┘
```

或者本地代理模式：

```
本机                             远程服务器 (192.168.2.xx)
┌─────────────────┐              ┌──────────────┐
│ Flask :18080    │  HTTP :11434 │  Ollama      │
│ (local_server)  │ ───────────→ │              │
│        ▲        │              └──────────────┘
│ localhost:18080 │
│   浏览器        │
└─────────────────┘
```

In [ ]:
import requests
import json

def test_gateway(url="http://192.168.2.xx:11434"):
    """测试 Ollama 网关连通性"""
    try:
        r = requests.get(f"{url}/api/tags", timeout=5)
        models = r.json().get("models", [])
        print(f"网关状态: 在线")
        print(f"可用模型: {len(models)} 个")
        for m in models:
            name = m.get("name", "?")
            size_mb = m.get("size", 0) / 1024 / 1024
            print(f"  - {name:30s} {size_mb:.1f}MB")
        return True
    except Exception as e:
        print(f"网关状态: 离线 ({e})")
        return False

# 不实际调用（需要在能连通远程服务器的环境运行）
print("网关测试函数已定义。")
print("在能连通远程服务器的机器上运行: test_gateway()")

## 最终构建：部署状态检查脚本

把前面学到的所有概念组合成一个一键检查脚本。

In [ ]:
import os
import sys
import time
import requests

class DeploymentChecker:
    """LumiLearn 部署状态检查器"""

    def __init__(self, gateway_url="http://192.168.2.xx:11434",
                 terminal_url="http://localhost:18080"):
        self.gateway_url = gateway_url
        self.terminal_url = terminal_url
        self.results = []

    def check(self, name, fn):
        try:
            ok, msg = fn()
            status = "✓" if ok else "✗"
            self.results.append((status, name, msg))
        except Exception as e:
            self.results.append(("✗", name, str(e)))

    def check_gateway(self):
        t0 = time.time()
        r = requests.get(f"{self.gateway_url}/api/tags", timeout=5)
        latency = int((time.time() - t0) * 1000)
        models = r.json().get("models", [])
        names = [m["name"] for m in models]
        return True, f"{len(models)} models, {latency}ms | {', '.join(names[:3])}"

    def check_terminal(self):
        r = requests.get(f"{self.terminal_url}/health", timeout=5)
        data = r.json()
        return data["status"] == "healthy", f"{data['status']}, gateway={data.get('gateway','?')}"

    def check_html(self):
        r = requests.get(f"{self.terminal_url}/", timeout=5)
        has_title = "LumiTerminal" in r.text
        return has_title, f"HTML {len(r.text)} chars"

    def check_chat(self, model="qwen2.5:7b"):
        r = requests.post(f"{self.terminal_url}/api/chat", json={
            "model": model,
            "messages": [{"role": "user", "content": "hi"}],
            "stream": False
        }, timeout=30)
        content = r.json().get("message", {}).get("content", "")
        return len(content) > 0, f"model={model}, response_len={len(content)}"

    def run_all(self):
        print("=" * 50)
        print("  LumiLearn 部署状态检查")
        print("=" * 50)
        print(f"Gateway:  {self.gateway_url}")
        print(f"Terminal: {self.terminal_url}")
        print()

        self.check("Ollama 网关", self.check_gateway)
        self.check("终端 /health", self.check_terminal)
        self.check("终端 HTML", self.check_html)
        self.check("Chat API", self.check_chat)

        for status, name, msg in self.results:
            print(f"  {status} {name:15s} {msg}")

        passed = sum(1 for s, _, _ in self.results if s == "✓")
        total = len(self.results)
        print(f"\n结果: {passed}/{total} 通过")

        return passed == total

# 在能连通远程服务器的环境运行:
# checker = DeploymentChecker()
# checker.run_all()
print("DeploymentChecker 类已定义。")
print("在能连通远程服务器 + 终端运行的环境中使用:")
print("  checker = DeploymentChecker()")
print("  checker.run_all()")

## 练习与拓展

1. **修改 num_predict**: 在 Modelfile 中把 `num_predict` 改为 512，观察生成长度变化。
2. **对比量化效果**: 同时部署 FP32 和 Q4_K_M 版本，对比推理速度和质量。
3. **添加新模型**: 用 `ollama create` 注册一个带有不同 system prompt 的版本（如 `lumilearn-v5:math` 专攻数学）。
4. **监控延迟**: 修改 DeploymentChecker，添加平均延迟统计和折线图。
5. **修复 tokenizer Bug**: 如果 `tokenizer.ggml.tokens` 为 null，分析 `export_gguf_v5.py` 中哪里写入了空字符串占位符。

## 关键心得
- GGUF 的 `tokenizer.ggml.tokens` 是 Ollama 加载时第一个读取的字段，为空会导致模型无法使用
- BPE tokenizer 比字符级 tokenizer 节省约 50% token 数
- 统一 IP 不同端口架构让浏览器可直接访问，无需复杂的网络配置
- `gguf` Python 库比手动二进制写入更可靠